In [45]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


#Regresión Logistica
 Importar librerías necesarias

In [46]:
# se utiliza para el manejo de rutas y directorios.
import os

# Calculo cientifico y vectorial para python
import numpy as np

# Librerias para graficar
import matplotlib.pyplot as plt

# Modulo de optimización de scipy
from scipy import optimize

# le dice a matplotlib que incruste gráficos en el cuaderno
%matplotlib inline
# Libreria de pandas
import pandas as pd

In [47]:
# Cargar dataset
# Columnas del dataset (según la documentación del Adult Dataset en UCI)
# Columnas definidas en el dataset
columnas = [
    "age", "workclass", "fnlwgt", "education", "education-num",
    "marital-status", "occupation", "relationship", "race", "sex",
    "capital-gain", "capital-loss", "hours-per-week", "native-country", "income"
]

# Cargar dataset desde la URL de UCI
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
df = pd.read_csv(url, header=None, names=columnas, na_values=" ?", skipinitialspace=True)
print("Columnas:", df.columns)
print(df.head())
df.head()

Columnas: Index(['age', 'workclass', 'fnlwgt', 'education', 'education-num',
       'marital-status', 'occupation', 'relationship', 'race', 'sex',
       'capital-gain', 'capital-loss', 'hours-per-week', 'native-country',
       'income'],
      dtype='object')
   age         workclass  fnlwgt  education  education-num  \
0   39         State-gov   77516  Bachelors             13   
1   50  Self-emp-not-inc   83311  Bachelors             13   
2   38           Private  215646    HS-grad              9   
3   53           Private  234721       11th              7   
4   28           Private  338409  Bachelors             13   

       marital-status         occupation   relationship   race     sex  \
0       Never-married       Adm-clerical  Not-in-family  White    Male   
1  Married-civ-spouse    Exec-managerial        Husband  White    Male   
2            Divorced  Handlers-cleaners  Not-in-family  White    Male   
3  Married-civ-spouse  Handlers-cleaners        Husband  Black    Mal

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


# Preprocesamiento


In [48]:
from sklearn.preprocessing import LabelEncoder



# 1. Label encoding para income
le = LabelEncoder()
df["income"] = le.fit_transform(df["income"])  # <=50K → 0, >50K → 1



# Verificamos las primeras filas
df.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,0
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,0
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,0
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,0
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,0


In [49]:
print("Tipos de datos:\n", df.dtypes)
print(df.columns.tolist())


Tipos de datos:
 age                int64
workclass         object
fnlwgt             int64
education         object
education-num      int64
marital-status    object
occupation        object
relationship      object
race              object
sex               object
capital-gain       int64
capital-loss       int64
hours-per-week     int64
native-country    object
income             int64
dtype: object
['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income']


In [50]:
df = df.drop(columns=["fnlwgt", "native-country"])

# Convertir categóricas a dummies
df = pd.get_dummies(df, drop_first=True)
#volviendo los numeros a enteros
df = df.astype(int)

In [51]:
print(df)

       age  education-num  capital-gain  capital-loss  hours-per-week  income  \
0       39             13          2174             0              40       0   
1       50             13             0             0              13       0   
2       38              9             0             0              40       0   
3       53              7             0             0              40       0   
4       28             13             0             0              40       0   
...    ...            ...           ...           ...             ...     ...   
32556   27             12             0             0              38       0   
32557   40              9             0             0              40       1   
32558   58              9             0             0              40       0   
32559   22              9             0             0              20       0   
32560   52              9         15024             0              40       1   

       workclass_Federal-go

In [52]:
# Variables predictoras y target
X = df.drop("income", axis=1).values
y = df["income"].values

num_labels = len(np.unique(y))
# Columnas numéricas (continuas + binarias)
# num_cols = ["age", "avg_glucose_level", "bmi", "hypertension", "heart_disease"]

# Columnas categóricas (todo lo demás excepto 'id')
# cat_cols = [col for col in X.columns if col not in num_cols ]
print("Número de clases:", num_labels)

Número de clases: 2


In [53]:
before = df.shape[0]

# Eliminar filas con NaN
df_clean = df.dropna()

# Número de filas después de limpiar
after = df_clean.shape[0]

# Calcular cuántos ejemplos se borraron
removed = before - after

print(f"Filas antes: {before}")
print(f"Filas después: {after}")
print(f"Filas eliminadas: {removed}")

Filas antes: 32561
Filas después: 32561
Filas eliminadas: 0


In [54]:
print(X)
print(y)

[[   39    13  2174 ...     0     1     1]
 [   50    13     0 ...     0     1     1]
 [   38     9     0 ...     0     1     1]
 ...
 [   58     9     0 ...     0     1     0]
 [   22     9     0 ...     0     1     1]
 [   52     9 15024 ...     0     1     0]]
[0 0 0 ... 0 0 1]


# Normalización de los datos
→ Escalar las variables numéricas para que tengan media 0 y desviación estándar 1 (o entre 0 y 1). Esto ayuda al descenso por gradiente.

In [55]:
# 3. Normalización
def featureNormalize(X):
    mu = X.mean(axis=0)
    sigma = X.std(axis=0)
    X_norm = (X - mu) / sigma
    return X_norm, mu, sigma

X_norm, mu, sigma = featureNormalize(X)

In [56]:
print(X_norm[0,:])
print(y)

[ 0.03067056  1.13473876  0.1484529  -0.21665953 -0.03542945 -0.17429511
 -0.26209736 -0.01466381 -1.5167923  -0.18838933 -0.29093568  4.90769968
 -0.02073999 -0.19348662 -0.11609195 -0.07201601 -0.10164955 -0.1422718
 -0.12664495 -0.18406376 -0.21053433  2.25399324 -0.11334387 -0.68994199
 -0.23637391 -0.03960742 -0.13419553 -0.53714425 -0.02658695 -0.92284068
 -0.11403678  1.43105786 -0.1802846  -0.17735813  2.76348874 -0.01662771
 -0.37949517 -0.37774555 -0.17745022 -0.20957797 -0.25595432 -0.33554133
 -0.06780164 -0.38166338 -0.14260848 -0.35531609 -0.17127887 -0.22710355
  1.70899099 -0.17624972 -0.42934582 -0.34403232 -0.22492681 -0.18155194
 -0.32576824 -0.09161163  0.4130197   0.70307135]
[0 0 0 ... 0 0 1]


# División de datos en entrenamiento (80%) y prueba (20%)

In [57]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X_norm, y, test_size=0.2, random_state=42, stratify=y
)

print("Tamaño entrenamiento:", X_train.shape[0])
print("Tamaño prueba:", X_test.shape[0])

Tamaño entrenamiento: 26048
Tamaño prueba: 6513


#Definir funciones

In [58]:
def sigmoid(z):
    # Calcula la sigmoide de una entrada z
    # convierte la entrada a un arreglo numpy
    z = np.array(z)
    g = np.zeros(z.shape)
    g = 1 / (1 + np.exp(-z))
    return g

In [59]:
def calcularCosto(theta, X_train, y_train):
    # Inicializar algunos valores utiles
    m = y.size  # numero de ejemplos de entrenamiento

    J = 0
    h = sigmoid(X_train.dot(theta.T))
    J = (1 / m) * np.sum(-y.dot(np.log(h)) - (1 - y).dot(np.log(1 - h)))

    return J

In [60]:
def descensoGradiente(theta, X_train, y_train, alpha, num_iters):
    # Inicializa algunos valores
    m = y_train.shape[0] # numero de ejemplos de entrenamiento

    # realiza una copia de theta, el cual será acutalizada por el descenso por el gradiente
    theta = theta.copy()
    J_history = []

    for i in range(num_iters):
        h = sigmoid(X_train.dot(theta))  # (m,)
      # actualización de parámetros
        theta = theta - (alpha / m) * (X_train.T.dot(h - y_train))

        J_history.append(calcularCosto(theta, X_train, y_train))
    return theta, J_history

In [61]:
def lrCostFunction(theta, X_train, y_train, lambda_):
    """
    Calcula el costo de usar theta como parámetro para la regresión logística regularizada y
    el gradiente del costo w.r.t. a los parámetros.

    Parametros
    ----------
    theta : array_like
        Parametro theta de la regresion logistica. Vector de la forma(shape) (n, ). n es el numero de caracteristicas
        incluida la intercepcion

    X : array_like
        Dataset con la forma(shape) (m x n). m es el numero de ejemplos, y n es el numero de
        caracteristicas (incluida la intercepcion).

    y_train : array_like
        El conjunto de etiquetas. Un vector con la forma (shape) (m, ). m es el numero de ejemplos

    lambda_ : float
        Parametro de regularización.

    Devuelve
    -------
    J : float
        El valor calculado para la funcion de costo regularizada.

    grad : array_like
        Un vector de la forma (shape) (n, ) que es el gradiente de la
        función de costo con respecto a theta, en los valores actuales de theta..
    """
#     alpha = 0.003
#     theta = theta.copy()
    # Inicializa algunos valores utiles
    m = y_train.size

    # convierte las etiquetas a valores enteros si son boleanos
    if y_train.dtype == bool:
        y_train = y_train.astype(int)

    J = 0
    grad = np.zeros(theta.shape)

    h = sigmoid(X_train.dot(theta.T))
    eps = 1e-10
    h = np.clip(h, eps, 1 - eps)
    temp = theta
    temp[0] = 0

#     J = (1 / m) * np.sum(-y.dot(np.log(h)) - (1 - y).dot(np.log(1 - h)))
    J = (1 / m) * np.sum(-y_train.dot(np.log(h)) - (1 - y_train).dot(np.log(1 - h))) + (lambda_ / (2 * m)) * np.sum(np.square(temp))

    grad = (1 / m) * (h - y_train).dot(X_train)
#     theta = theta - (alpha / m) * (h - y).dot(X)
    grad = grad + (lambda_ / m) * temp

    return J, grad
#    return J, theta

In [62]:
def OneVsAll(X_train, y_train, num_labels, lambda_):
    alpha = 0.01
    num_iters = 10000  # puedes ajustar si ves que tarda mucho

    m, n = X_train.shape
    all_theta = np.zeros((num_labels, n + 1))

    # Agregar columna de 1s (bias) a X_train
    X_train = np.concatenate([np.ones((m, 1)), X_train], axis=1)

    for c in np.arange(num_labels):
        # Inicializar parámetros
        initial_theta = np.zeros(n + 1)

        # Vector binario: 1 si y==c, 0 si no
        y_actual = np.where(y_train == c, 1, 0)

        # Entrenar con descenso de gradiente
        theta, J_history = descensoGradiente(initial_theta, X_train, y_actual, alpha, num_iters)

        all_theta[c] = theta

        # Graficar convergencia del costo
        plt.plot(np.arange(len(J_history)), J_history, lw=2, label=f'Clase {c}')

    plt.xlabel('Número de iteraciones')
    plt.ylabel('Costo J')
    plt.legend()
    plt.title('Convergencia del costo para todas las clases')
    plt.show()

    return all_theta


In [63]:
def OneVsAllOM(X_train, y_train, num_labels, lambda_):
    """
    Entrena num_labels clasificadores de regresión logística
    y devuelve cada clasificador en una matriz all_theta.
    """

    m, n = X_train.shape
    all_theta = np.zeros((num_labels, n + 1))

    # Agrega columna de 1s (bias term)
    X_train_biased = np.concatenate([np.ones((m, 1)), X_train], axis=1)


    for c in np.arange(num_labels):
        initial_theta = np.zeros(n + 1)

        # y_actual = vector binario (1 si es clase c, 0 si no)
        y_actual = (y_train == c).astype(int)

        # Optimización con scipy
        res = optimize.minimize(
            lrCostFunction,           # función de costo + gradiente
            initial_theta,            # theta inicial
            args=(X_train_biased, y_actual, lambda_),  # parámetros extra
            jac=True,                 # devuelve gradiente
            method='CG',              # conjugate gradient
            options={'maxiter': 50}   # máximo de iteraciones
        )

        all_theta[c] = res.x

    return all_theta

In [64]:
lambda_ = 0.1
all_theta = OneVsAllOM(X_train, y_train, num_labels, lambda_)
print(all_theta.shape)

(2, 59)


In [65]:
print(all_theta)

[[ 2.07403462e+00 -3.55162882e-01 -5.75679259e-01 -2.31516571e+00
  -2.46841445e-01 -3.71351838e-01 -1.72100961e-01 -6.42768301e-02
   6.85704977e-02 -2.15432275e-01 -1.22492495e-01 -2.48815290e-04
  -2.78590340e-02  1.32329473e-01  3.91299751e-03 -1.69279185e-02
  -9.75714691e-03 -5.20752384e-02 -5.94905243e-03 -4.27833709e-03
  -9.67951782e-03 -5.32007463e-02 -1.82253298e-01 -7.77773106e-02
  -1.11363442e-01 -1.24021146e-01  4.71590232e-01 -1.17363493e-01
  -1.58269719e-01 -5.78974013e-02 -1.06182306e+00  6.24080089e-04
   1.85258309e-01  3.26631873e-02 -4.46018182e-02 -4.33122320e-02
   9.13380558e-02 -7.99232166e-02 -3.06002950e-01  1.48906868e-01
   9.71089794e-02  3.89058208e-02  1.88177089e-01  2.68400547e-01
  -2.31231173e-01 -1.05965947e-01 -1.29932069e-01 -1.32954000e-01
  -1.52509439e-02 -1.95959088e-01  8.38652288e-02  2.77101056e-01
  -1.02431676e-01 -2.91432819e-01 -9.58369279e-02 -1.54195333e-01
  -1.35356561e-02 -2.49671864e-01 -3.98782661e-01]
 [-2.07403462e+00  3.5516

In [66]:
def predictOneVsAll(all_theta, X):
    """
    Devuelve un vector de predicciones para cada ejemplo en la matriz X.
    Tenga en cuenta que X contiene los ejemplos en filas.
    all_theta es una matriz donde la i-ésima fila es un vector theta de regresión logística entrenada para la i-ésima clase.
    Debe establecer p en un vector de valores de 0..K-1 (por ejemplo, p = [0, 2, 0, 1]
    predice clases 0, 2, 0, 1 para 4 ejemplos).

    Parametros
    ----------
    all_theta : array_like
        The trained parameters for logistic regression for each class.
        This is a matrix of shape (K x n+1) where K is number of classes
        and n is number of features without the bias.

    X : array_like
        Data points to predict their labels. This is a matrix of shape
        (m x n) where m is number of data points to predict, and n is number
        of features without the bias term. Note we add the bias term for X in
        this function.

    Devuelve
    -------
    p : array_like
        The predictions for each data point in X. This is a vector of shape (m, ).
    """

    m = X.shape[0];
    num_labels = all_theta.shape[0]
    p = np.zeros(m)
    # Agrega columna de 1s (bias) a X
    Xb = np.concatenate([np.ones((m, 1)), X], axis=1)

    # Calcula probabilidades para cada clase y toma la máxima
    p = np.argmax(sigmoid(Xb.dot(all_theta.T)), axis=1)

    return p

In [67]:
# Predicciones sobre entrenamiento
pred_train = predictOneVsAll(all_theta, X_train)

# Predicciones sobre prueba
pred_test = predictOneVsAll(all_theta, X_test)

# Precisión
print("Precisión entrenamiento: {:.2f}%".format(np.mean(pred_train == y_train) * 100))
print("Precisión prueba: {:.2f}%".format(np.mean(pred_test == y_test) * 100))


Precisión entrenamiento: 85.06%
Precisión prueba: 85.46%


Precisión entrenamiento: 85.06%
El 85.06% de los ejemplos del conjunto de entrenamiento fueron correctamente clasificados por el modelo.

Precisión prueba: 85.46%
El 85.46% de los ejemplos del conjunto de prueba fueron correctamente clasificados.
